### Risk Evaluation Criteria
Trigger alert in N8N if:

- Daily cases increase for 3 consecutive days
- Test positivity rate > defined threshold
- Low vaccination growth + rising cases


```SQL
WITH history AS (
    SELECT 
        state,
        date,
        daily_new_cases,
        positive_test_rate,
        population,
        vaccination_rate,
        
        LAG(daily_new_cases, 1) OVER (PARTITION BY state ORDER BY date) as cases_1_day_ago,
        LAG(daily_new_cases, 2) OVER (PARTITION BY state ORDER BY date) as cases_2_days_ago,
        LAG(daily_new_cases, 3) OVER (PARTITION BY state ORDER BY date) as cases_3_days_ago,
        
        vaccination_rate - LAG(vaccination_rate, 1) OVER (PARTITION BY state ORDER BY date) AS daily_vax_growth_pct

    FROM covid_summary_clean
    WHERE date >= '2021-07-01' 
),
risk_analysis AS (
    SELECT 
        *,
        -- RISK A: 3 Consecutive Days of Rising Cases
        CASE 
            WHEN daily_new_cases > cases_1_day_ago 
             AND cases_1_day_ago > cases_2_days_ago 
             AND cases_2_days_ago > cases_3_days_ago 
            THEN TRUE ELSE FALSE 
        END as risk_consecutive_rise,

        -- RISK B: High Positivity Rate (> 10%)
        CASE 
            WHEN positive_test_rate > 10 THEN TRUE ELSE FALSE 
        END as risk_high_positivity,

        -- RISK C: Vaccination Stalled & High Volume Cases (> 500)
        CASE 
            WHEN daily_vax_growth_pct < 0.1 
             AND daily_new_cases > cases_1_day_ago 
             AND daily_new_cases > 500
            THEN TRUE ELSE FALSE 
        END as risk_vax_stall
    FROM history
)
SELECT 
    state,
    date,
    daily_new_cases,
    positive_test_rate,       
    population,              
    vaccination_rate,        
    CONCAT_WS(' | ', 
        CASE WHEN risk_consecutive_rise THEN '⚠️ Outbreak Trend: Cases rising for 3 consecutive days' END,
        CASE WHEN risk_high_positivity THEN '⚠️ High Positivity: Rate > 10%' END,
        CASE WHEN risk_vax_stall THEN '⚠️ Vulnerability: Vaccination stalled while cases are rising' END
    ) as risk_summary
FROM risk_analysis
WHERE 
    date = '2021-08-11'
    AND (risk_consecutive_rise OR risk_high_positivity OR risk_vax_stall);

##### There were many states with Risk -> Vulnerability: Vaccination stalled while cases are rising
To confirm authenticity we are using this debug script

```SQL
WITH debug_history AS (
    SELECT 
        state,
        date,
        daily_new_cases,
        vaccination_rate,
        -- Get Yesterday's Rate
        LAG(vaccination_rate, 1) OVER (PARTITION BY state ORDER BY date) as prev_vax_rate
    FROM covid_summary_clean
    -- Just look at August
    WHERE date >= '2021-08-01' AND date <= '2021-08-12'
)
SELECT 
    state,
    daily_new_cases,
    prev_vax_rate,
    vaccination_rate as current_vax_rate,
    -- The calculated metric
    (vaccination_rate - prev_vax_rate) as daily_growth_diff,
    
    -- Check if it triggers your filter
    CASE 
        WHEN (vaccination_rate - prev_vax_rate) < 0.005 THEN '🚩 FLAG' 
        ELSE 'OK' 
    END as status
FROM debug_history
WHERE date = '2021-08-11'
ORDER BY daily_growth_diff ASC;



##### Insights
- There  are many states with 0 vaccination rate improvement over time, while having high number of daily cases.
